# Naive Bayes
Bayesian statistics applied as a classification technique truly emerged in the 1960s to solve text information retrieval, though the underlying theory and early applications date back to the mid-1700s (Reverend Thomas Bayes). 

Naive Bayes is a generative statistical inference technique. Like other generative methods, it tries to estimate the Bayes optimal classifier by factoring the posterior into the likelihood and prior (see our repository's [README.md](../../../../README.md) and the classification [README.md](../../../README.md) for more). However, Naive Bayes takes this a step further by assuming its features are *class-conditionally independent*. This allows us to break down the Bayes optimal rule as follows:

$$
f(\vec{x}) = \text{argmax}_c (P(Y=y_c|\vec{x})) \\
P(Y=y_c|\vec{x}) = \frac{P(X=\vec{x}|y_c) \cdot P(y_c)}{P(\vec{x})} \\
P(X=\vec{x}|y_c) = P(x_1, x_2, ..., x_p | y_c) = \prod_{i=1}^p P(x_i|y_c)
$$

Here, we simply have to capture the independent class-conditional distributions for each feature individually and multiply them to estimate the joint distribution. In the real world, this is a heavy and often unrealistic bias, but it is computationally incredibly efficient.

Like our other generative methods, we can estimate the class prior $P(y_c)$ by its proportion in the training dataset. Since the marginal distribution of the sample $P(\vec{x})$ is a constant denominator across all classes, we can ignore it. Finally, by applying a log transformation (which is rank-preserving), we turn the multiplications into additions to prevent floating-point underflow and maintain computational integrity. This gives us our final, equivalent class score to optimize over:

$$
f(\vec{x}) = \text{argmax}_c \left( \log(P(y_c)) + \sum_{i=1}^p \log(P(x_i | y_c)) \right)
$$

In practice, we simply look at the data type of each individual feature to determine which likelihood distribution to apply. 

Because of our naive independence assumption, we don't have to calculate massive, multi-dimensional distributions. Instead:
*   For a **continuous feature**, we can assume it follows a single-variable (univariate) Gaussian distribution and compute its individual probability density $\text{pdf}_i(x_i)$.
*   For a **binary/boolean feature** (e.g., word presence), we use a simple Bernoulli probability $P(x_i | y_c)$.
*   For **count/frequency features** (e.g., word counts), we use a Multinomial probability.

Ultimately, we compute the log-likelihood for each feature independently, and simply sum them together along with the log of the class prior. This generates the final class score that we maximize to make our prediction.

For training we estimate the class specific distributions for each feature, then use those for inference.

### Guassian Features
The guassian distribution models continous random variables, and is common when measuring physical phenomena or asymptotic behavior of random variable parameters. For Naive Bayes we consider each continous random variable it's own class conditional gaussian:

$$
x \sim N(\mu, \sigma^2)
$$

It's likelihood is the following density function:

$$
\text{pdf}(x) = \frac{1}{\sqrt{2 \pi \sigma^2}} e^{\frac{-1}{2\sigma^2} \cdot (x - \mu)^2}
$$

Using it in our classifier:

$$
f(x) = \text{argmax}_c (\frac{1}{\sqrt{2 \pi \sigma_c^2}} e^{\frac{-1}{2\sigma_c^2} \cdot (x - \mu_c)^2} \cdot P(y_c))
$$

Applying the rank perserving log for stability:

$$
f(x) = \text{argmax}_c (\log(\frac{1}{\sqrt{2 \pi \sigma_c^2}}) + \frac{-1}{2\sigma_c^2} \cdot (x - \mu_c)^2 + \log(P(y_c)))
$$

In [45]:
import numpy as np

class_1_data = np.random.normal(5, 3, (1000, 1))
class_0_data = np.random.normal(-5,1, (1000, 1))

dataset = np.concat([np.concat([class_0_data, np.zeros(shape=(1000,1))], axis=-1),np.concat([class_1_data, np.ones(shape=(1000,1))], axis=-1)], axis=0)

np.random.shuffle(dataset)

training_dataset = dataset[:int(dataset.shape[0]*.8)]
test_dataset = dataset[int(dataset.shape[0]*.8):]

len(test_dataset), len(training_dataset)

(400, 1600)

In [46]:
class NaiveBayesClassifierGaussianBinarySingleVariable:
    def __init__(self):
        self.label_0_mean = None
        self.label_1_mean = None
        self.label_0_var = None
        self.label_1_var = None
        self.label_1_prior = None
        self.label_0_prior = None

    def train(self, dataset):
        '''
        Assumes last column is binary label
        '''
        label_1_data = dataset[dataset[:, -1] == 1][:, :-1]
        label_0_data = dataset[dataset[:, -1] == 0][:, :-1]

        self.label_1_mean = label_1_data.mean(axis=0)
        self.label_0_mean = label_0_data.mean(axis=0)

        self.label_1_var = label_1_data.var(axis=0)
        self.label_0_var = label_0_data.var(axis=0)

        self.label_1_prior = len(label_1_data) / len(dataset)
        self.label_0_prior = len(label_0_data) / len(dataset)


    def predict(self, data):
        if self.label_0_mean is None:
            raise Exception("Cannot predict without training.")
        class_1_score = np.log( 1 / np.sqrt(2*np.pi * self.label_1_var)) - 1 / (2 * self.label_1_var) * (data - self.label_1_mean)**2 + np.log(self.label_1_prior)
        class_0_score = np.log( 1 / np.sqrt(2*np.pi * self.label_0_var)) - 1 / (2 * self.label_0_var) * (data - self.label_0_mean)**2 + np.log(self.label_0_prior)
        return (class_1_score > class_0_score).astype(int).reshape(-1)

guassian_classifer = NaiveBayesClassifierGaussianBinarySingleVariable()
guassian_classifer.train(training_dataset)
print(guassian_classifer.__dict__)
(guassian_classifer.predict(test_dataset[:, :-1]) == test_dataset[:, -1]).mean()

{'label_0_mean': array([-5.01065661]), 'label_1_mean': array([4.83279707]), 'label_0_var': array([0.93410206]), 'label_1_var': array([8.98808863]), 'label_1_prior': 0.50375, 'label_0_prior': 0.49625}


np.float64(0.995)

We were able to recover the original data producing parameters, and classify with the near (as good as our estimate) bayes optimal rule.

### Bernoulli Features
Bernoulli distribution models binary outcomes with a probability that its 1 (by convention), the alternative is always $1 - p$.
If a feature $X$ follows a Bernoulli distribution with probability $p$, we denote this as:
$$
X \sim \text{Bernoulli}(p)
$$

1. The Piecewise Form (Logical)
The most intuitive way to write this PMF is as a logical map, evaluating the two possible discrete outcomes:
$$
P(X=x) = \begin{cases} 
p & \text{if } x = 1 \\
1-p & \text{if } x = 0 
\end{cases}
$$

2. The Closed Form (Algebraic)
To use this in machine learning (specifically so we can apply a log transformation), we collapse the piecewise logic into a single, compact algebraic equation using exponents:
$$
P(X=x) = p^x(1-p)^{1-x}
$$


So plugging this into our classifier (Maximum A Posteriori (MAP)):

$$
f(x) = \text{argmax}_c (p_c^x \cdot (1 - p_c)^{1-x} \cdot P(y_c))
$$

Where $p_c$ is the class conditional probability of the outcome of 1. With the usual log trick:

$$
f(x) = \text{argmax}_c (x \log(p_c) +(1-x) \log(1 - p_c) + \log(P(y_c)))
$$

In [54]:
class_0_data = np.random.binomial(1, 0.1, (1000, 1))
class_1_data = np.random.binomial(1, 0.8, (1000, 1))

dataset = np.concat([np.concat([class_0_data, np.zeros_like(class_0_data)], axis=1), np.concat([class_1_data, np.ones_like(class_1_data)], axis=1)],axis=0)
np.random.shuffle(dataset)

training_dataset = dataset[:int(dataset.shape[0]*0.8)]
test_dataset = dataset[int(dataset.shape[0]*0.8):]

class NaiveBayesClassifierBernoulliBinarySingleVariable():
    def __init__(self):
        self.class_0_p = None
        self.class_0_prior = None
        self.class_1_p = None
        self.class_1_prior = None
    def train(self, dataset):
        self.class_0_p = dataset[dataset[:, -1] == 0][:, :-1].mean()
        self.class_0_prior = len( dataset[dataset[:, -1] == 0]) / len(dataset)

        self.class_1_p = dataset[dataset[:, -1] == 1][:, :-1].mean()
        self.class_1_prior = len( dataset[dataset[:, -1] == 1]) / len(dataset)

    def predict(self, data):
        if self.class_0_p is None:
            raise Exception("Cannot predict without training")
        class_0_score = np.log(self.class_0_prior) + data * np.log(self.class_0_p) + (1 - data) * np.log(1 - self.class_0_p)
        class_1_score = np.log(self.class_1_prior) + data * np.log(self.class_1_p) + (1 - data) * np.log(1 - self.class_1_p)
        return (class_1_score > class_0_score).reshape(-1)

model = NaiveBayesClassifierBernoulliBinarySingleVariable()
model.train(training_dataset)
print(model.__dict__)
(model.predict(test_dataset[:, :-1]) == test_dataset[:, -1]).mean()

{'class_0_p': np.float64(0.10984848484848485), 'class_0_prior': 0.495, 'class_1_p': np.float64(0.8032178217821783), 'class_1_prior': 0.505}


np.float64(0.87)

We were able to recover the original data producing parameters, and classify with the near (as good as our estimate) bayes optimal rule. To prevent overflow in the eventual application we may want to add laplace smoothing (+1 to prior and probability counts to avoid $\log(0)$).

### Multinomial Features
The multinomial distribution assigns a probability to a vector of event counts (integers), $P(\vec{x})$, using a probability mass function based on the outcome probabilities of single events ($\vec{p}$). It does this using combinatorics:

$$
P(\vec{x}) = \frac{N!}{\prod_{i=1}^p x_i!} \cdot \prod_{i=1}^p p_i^{x_i}
$$

This scales the probability of seeing that exact quantity of events by the number of combinatorial ways to arrange those events.

Because this combinatorial weight $\left( \frac{N!}{\prod_{i=1}^p x_i!} \right)$ (which is the unique ways to arrange the sequence of the events (total! / repeating!)) depends only on the new document and is constant across all classes, it can be factored out and ignored when maximizing the class conditional likelihood. After applying a log transformation (which is rank-preserving/monotonic) for computational stability, we get our equivalent function to maximize:

$$
f(\vec{x}) = \text{argmax}_c (P(\vec{x} | y_c) \cdot P(y_c)) \\
= \text{argmax}_c \left( \frac{N!}{\prod_{i=1}^p x_i!} \cdot \prod_{i=1}^p p_{ci}^{x_i} \cdot P(y_c) \right) \\
= \text{argmax}_c \left( \sum_{i=1}^p x_i \log(p_{ci}) + \log(P(y_c)) \right)
$$

To get the learned parameter $\vec{p}_c$ during training:

$$
p_{ic} = \frac{N_{ic}}{N_c}
$$

Here, you treat all of a class's samples as one giant pooled event, and calculate the proportion (from base all feature counts summed up for a given class) of each feature's frequency to get its probability. 

When you treat a set of features this way, you are assuming that their specific order does not matter (e.g., the "bag-of-words" model), as implicit in the multinomial distribution.

In [55]:
# coming soon

(Tk), examples for each

(Tk), combine theory + example

(Tk), classic goals with two datasets, tests for assumpttions for interpertabilty ()